# PySpark End-to-End Example: Retail Sales Analysis

**Runs directly in Google Colab — no file uploads needed.** All the sample data is created inline in the notebook itself.

## Scenario

You're a data analyst at **UrbanCart**, a mid-size retail chain with stores in several regions. Leadership wants answers to a few business questions from last quarter's transaction data:

1. What was our total revenue, and how is it split by product category?
2. Which region is performing best?
3. Who are our top 5 customers by spend?
4. What's the monthly revenue trend?
5. Which orders were "large" orders (above $500) that deserve a closer look?

We'll walk through the full pipeline: **create a Spark session → load data → clean/transform → join → analyze → save results** — the same shape as a real production job, just at a small scale so it runs instantly.


## Step 0 — Install PySpark

Run this cell first. Google Colab already has Java installed, so this is the only setup step needed.


In [ ]:
!pip install pyspark -q
print("PySpark installed.")


## Step 1 — Create a SparkSession

This is the entry point to all Spark functionality.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("UrbanCart_Sales_Analysis")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
spark


## Step 2 — The Data

No file upload needed — we create two small tables directly in Python and turn them into Spark DataFrames:

- **orders**: one row per transaction (order id, customer, product, category, quantity, price, date, region)
- **customers**: a small customer lookup table (customer id, name, signup tier)

In a real job these would come from `spark.read.csv(...)`, a database, or a data lake — the rest of the pipeline is identical either way.


In [ ]:
orders_data = [
    (1001, "C01", "Wireless Mouse",     "Electronics", 2,  25.00, "2024-01-05", "West"),
    (1002, "C02", "Office Chair",       "Furniture",   1, 180.00, "2024-01-06", "East"),
    (1003, "C03", "Laptop Stand",       "Electronics", 1,  45.00, "2024-01-08", "West"),
    (1004, "C01", "4K Monitor",         "Electronics", 1, 320.00, "2024-01-12", "West"),
    (1005, "C04", "Standing Desk",      "Furniture",   1, 450.00, "2024-01-15", "North"),
    (1006, "C05", "Desk Lamp",          "Furniture",   3,  20.00, "2024-01-18", "South"),
    (1007, "C02", "Bluetooth Speaker",  "Electronics", 2,  60.00, "2024-02-02", "East"),
    (1008, "C06", "Ergonomic Keyboard", "Electronics", 1,  85.00, "2024-02-05", "North"),
    (1009, "C03", "Bookshelf",          "Furniture",   1, 150.00, "2024-02-09", "West"),
    (1010, "C01", "Webcam",             "Electronics", 1,  70.00, "2024-02-14", "West"),
    (1011, "C07", "Office Chair",       "Furniture",   2, 180.00, "2024-02-20", "South"),
    (1012, "C04", "Wireless Mouse",     "Electronics", 3,  25.00, "2024-02-22", "North"),
    (1013, "C05", "4K Monitor",         "Electronics", 2, 320.00, "2024-03-01", "South"),
    (1014, "C08", "Standing Desk",      "Furniture",   1, 450.00, "2024-03-04", "East"),
    (1015, "C02", "Laptop Stand",       "Electronics", 2,  45.00, "2024-03-09", "East"),
    (1016, "C06", "Bookshelf",          "Furniture",   2, 150.00, "2024-03-12", "North"),
    (1017, "C01", "Standing Desk",      "Furniture",   1, 450.00, "2024-03-15", "West"),
    (1018, "C07", "Bluetooth Speaker",  "Electronics", 1,  60.00, "2024-03-19", "South"),
    (1019, "C03", "Ergonomic Keyboard", "Electronics", 2,  85.00, "2024-03-22", "West"),
    (1020, "C08", "4K Monitor",         "Electronics", 1, 320.00, "2024-03-27", "East"),
]
orders_columns = ["order_id", "customer_id", "product", "category", "quantity", "unit_price", "order_date", "region"]

customers_data = [
    ("C01", "Ava Thompson",  "Gold"),
    ("C02", "Liam Chen",     "Silver"),
    ("C03", "Sofia Martinez","Gold"),
    ("C04", "Noah Patel",    "Bronze"),
    ("C05", "Mia Johnson",   "Silver"),
    ("C06", "Ethan Wright",  "Bronze"),
    ("C07", "Grace Kim",     "Gold"),
    ("C08", "Lucas Rossi",   "Silver"),
]
customers_columns = ["customer_id", "customer_name", "loyalty_tier"]

orders = spark.createDataFrame(orders_data, orders_columns)
orders = orders.withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))

customers = spark.createDataFrame(customers_data, customers_columns)

print("Orders:")
orders.show(5)
print("Customers:")
customers.show()


## Step 3 — Clean & Transform

Add a `total_amount` column and check for any data quality issues (a good habit on every real pipeline, even when — like here — the data is already clean).


In [ ]:
# Data quality check: any nulls, or non-positive quantity/price?
issues = orders.filter(
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    (F.col("quantity") <= 0) |
    (F.col("unit_price") <= 0)
)
print("Rows with data quality issues:", issues.count())

# Add a derived column: total_amount = quantity * unit_price
orders = orders.withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
orders = orders.withColumn("order_month", F.date_format("order_date", "yyyy-MM"))

orders.select("order_id", "product", "quantity", "unit_price", "total_amount", "order_month").show(5)


## Step 4 — Join Orders With Customer Info


In [ ]:
sales = orders.join(customers, on="customer_id", how="left")
sales.select("order_id", "customer_name", "loyalty_tier", "product", "total_amount", "region").show(5)


## Step 5 — Answer the Business Questions

### Q1: Total revenue, and revenue by category


In [ ]:
total_revenue = sales.agg(F.round(F.sum("total_amount"), 2).alias("total_revenue")).collect()[0]["total_revenue"]
print(f"Total revenue: ${total_revenue:,.2f}")

revenue_by_category = (
    sales.groupBy("category")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("revenue").desc())
)
revenue_by_category.show()


### Q2: Revenue by region — which region is performing best?


In [ ]:
revenue_by_region = (
    sales.groupBy("region")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.countDistinct("customer_id").alias("unique_customers"),
    )
    .orderBy(F.col("revenue").desc())
)
revenue_by_region.show()


### Q3: Top 5 customers by total spend


In [ ]:
top_customers = (
    sales.groupBy("customer_id", "customer_name", "loyalty_tier")
    .agg(F.round(F.sum("total_amount"), 2).alias("total_spend"))
    .orderBy(F.col("total_spend").desc())
    .limit(5)
)
top_customers.show()


### Q4: Monthly revenue trend (with running total)

This uses a **window function** — it computes a running total across months without collapsing the rows, so we can see both the monthly figure and the cumulative figure side by side.


In [ ]:
monthly_revenue = (
    sales.groupBy("order_month")
    .agg(F.round(F.sum("total_amount"), 2).alias("monthly_revenue"))
    .orderBy("order_month")
)

running_window = Window.orderBy("order_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
monthly_with_running = monthly_revenue.withColumn(
    "running_total", F.round(F.sum("monthly_revenue").over(running_window), 2)
)
monthly_with_running.show()


### Q5: Flag large orders (> $500) for review


In [ ]:
large_orders = sales.filter(F.col("total_amount") > 500).select(
    "order_id", "customer_name", "product", "quantity", "total_amount", "region", "order_date"
).orderBy(F.col("total_amount").desc())

large_orders.show()
print("Number of large orders:", large_orders.count())


## Step 6 — Save the Results

In a real pipeline you'd write this to a data lake (Parquet) or a warehouse table. Here we save the category summary as both CSV and Parquet so you can see the output on Colab's local disk (`/content/`).


In [ ]:
revenue_by_category.write.mode("overwrite").option("header", True).csv("/content/revenue_by_category_csv")
revenue_by_category.write.mode("overwrite").parquet("/content/revenue_by_category_parquet")

print("Saved output to /content/revenue_by_category_csv and /content/revenue_by_category_parquet")
print("(In Colab, use the Files panel on the left to browse and download these.)")


## Step 7 — Also Try It With Plain SQL

Everything above can equally be written as SQL against a temp view — same engine, same performance, just different syntax.


In [ ]:
sales.createOrReplaceTempView("sales")

spark.sql("""
    SELECT category,
           ROUND(SUM(total_amount), 2) AS revenue,
           COUNT(order_id) AS num_orders
    FROM sales
    GROUP BY category
    ORDER BY revenue DESC
""").show()


## Summary

In one notebook, you:
- Created a `SparkSession` and inline sample DataFrames (no file upload required)
- Cleaned data and added derived columns (`withColumn`)
- Joined two DataFrames (`orders` + `customers`)
- Aggregated with `groupBy().agg()`
- Used a window function for a running total
- Filtered for a business rule ("large orders")
- Wrote results out to CSV and Parquet
- Ran the same analysis with Spark SQL

This is the same shape as a real production Spark job — the only difference at real scale is that `orders`/`customers` would be read from a data lake or database instead of created inline, and the cluster would have many machines instead of just your Colab VM's cores.


In [ ]:
spark.stop()
print("Spark session stopped.")
